# Outputs, brought to life

A visual tour of a twelve-turbine wind farm: discover which turbines lose power, watch wakes turn with the wind, and connect the flow field to turbine performance.

This is a gallery, not an API catalogue. The [Outputs reference](../outputs.rst) covers output selection, result keys, aggregation, tables and file writers.

In [ ]:
%matplotlib inline
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import foxes
import foxes.constants as FC
import foxes.models.outputs as fop
import foxes.variables as FV

## A small farm, a full turn of the wind

We combine 72 wind-from directions with four speeds, giving **288 independent steady states**. Synthetic probabilities favour westerly winds. These are illustrative weights, not a measured climate or a chronological time series. Direction follows the meteorological convention: 0 degrees is north, 90 east and 270 west.

The 4-by-3 layout has 756 m spacing (six rotor diameters). A Gaussian velocity wake and an added-turbulence model make the two effects visible separately.

In [ ]:
# create wind data:
directions = np.arange(0, 360, 5)
speeds = np.array([6.0, 8.0, 10.0, 12.0])
direction_weights = np.exp(2 * np.cos(np.deg2rad(directions - 270))) + 0.25
weights = (direction_weights[:, None] * [0.15, 0.30, 0.35, 0.20]).ravel()
wind_data = pd.DataFrame(
    {
        FV.WD: np.repeat(directions, len(speeds)),
        FV.WS: np.tile(speeds, len(directions)),
        FV.WEIGHT: weights / weights.sum(),
    }
)
display(wind_data)

# create inflow states from wind data:
states = foxes.input.states.StatesTable(
    wind_data,
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    fixed_vars={FV.TI: 0.06, FV.RHO: 1.225},
)

# create wind farm:
farm = foxes.WindFarm()
for row, y in enumerate([-756.0, 0.0, 756.0]):
    for col, x in enumerate([0.0, 756.0, 1512.0, 2268.0]):
        farm.add_turbine(
            foxes.Turbine(
                xy=[x, y],
                turbine_models=["NREL5MW"],
                name=f"T{4 * row + col:02d}",
            ),
            verbosity=0,
        )

# create algorithm:
algo = foxes.algorithms.Downwind(
    farm,
    states,
    rotor_model="centre",
    wake_models=["Bastankhah2014_linear_k004", "CrespoHernandez_quadratic_k004"],
    verbosity=0,
)

# setup the engine:
engine = foxes.Engine.new(
    "process", chunk_size_states=50, chunk_size_points=4096, verbosity=1
)

# select a subset of states with 10 m/s wind speed, for later:
ten_ms_cases = wind_data.index[wind_data[FV.WS] == 10.0].tolist()

Calculate the farm once and keep its results for all subsequent views. The model book output `farm_results` is based on the default farm variable set and provides all inputs needed for flow calculations. The analysis outputs request derived efficiency/capacity metrics automatically; `ret_dict=True` keeps the result names explicit.

`P` is waked power, `AMB_P` is the no-wake reference and `EFF = P / AMB_P` is dimensionless. The all-turbine mean below is a ratio of weighted mean powers, not an average of turbine efficiencies.

In [ ]:
with engine:
    results = algo.calc_farm(
        outputs=[
            "farm_results_all",
            "farm_results_analysis_mean",
            "total_results_analysis_mean",
        ],
        ret_dict=True,
    )

farm_results = results[FC.FARM_RESULTS]
total_mean = results[FC.MEAN_TOTAL_FARM_RESULTS]
print("Available results:", ", ".join(results))
print(f"Mean total farm efficiency: {total_mean[FV.EFF].item():.1%}")

## 1. Visualizing data at turbines

Two views of the same results: the layout colours each turbine by its **weighted mean per-state efficiency**, while the heat map isolates 10 m/s and reveals the direction-dependent wake alignments. Both use the same dimensionless colour scale. Turbine indices stay in their original farm order, even when the wind reverses.

Passing our own axes lets different output models share a figure. This `calc_outputs` call reuses `farm_results`; it does not rerun farm physics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
with engine:
    dashboard = algo.calc_outputs(
        outputs=[
            fop.FarmLayoutOutput(
                result_name="layout",
                parameters=dict(
                    fig=fig,
                    ax=axes[0],
                    color_by=f"mean_{FV.EFF}",
                    cmap="viridis",
                    vmin=0.4,
                    vmax=1.0,
                    s=160,
                    anno_dely=100,
                    title="Weighted mean efficiency [1]",
                ),
            ),
            fop.StateTurbineMap(
                variable=FV.EFF,
                result_name="efficiency_map",
                isel={FC.STATE: ten_ms_cases},
                plot_pars=dict(
                    ax=axes[1],
                    cmap="inferno",
                    vmin=0.4,
                    vmax=1.0,
                    title="Efficiency at 10 m/s",
                    cbar_label="Efficiency [1]",
                ),
            ),
        ],
        farm_results=farm_results,
        ret_dict=True,
    )

axes[0].margins(0.18)
axes[1].set_xticks(np.arange(0, len(directions), 12) + 0.5, directions[::12])
axes[1].set_yticks(np.arange(algo.n_turbines) + 0.5, np.arange(algo.n_turbines))
axes[1].set_xlabel("Wind-from direction [deg]")
fig.tight_layout()
plt.show()
plt.close(fig)

## 2. Wind rose type visualizations

The left rose shows the ambient wind frequencies; its stacked bars sum to 100%. The right rose shows **mean efficiency in each direction/speed bin** at corner turbine 11. In this synthetic setup a common wind direction coincides with a strong wake loss. Radial position means wind speed on the right, but frequency on the left.

Rose outputs return `(axes, dataset)`: the plotted frequencies and binned values remain available in `roses["wind"][1]` and `roses["efficiency"][1]`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), subplot_kw={"projection": "polar"})
with engine:
    roses = algo.calc_outputs(
        outputs=[
            fop.RosePlotOutput(
                result_name="wind",
                wd_sectors=24,
                ws_bins=[0, 7, 9, 11, 13],
                plot_pars=dict(
                    ax=axes[0],
                    title="Ambient wind climate",
                    cmap="viridis",
                    freq_delta=2,
                    legend_pars=dict(loc="lower left", bbox_to_anchor=(-0.25, -0.15)),
                ),
            ),
            fop.WindRoseBinPlot(
                variable=FV.EFF,
                result_name="efficiency",
                turbine=11,
                wd_sectors=24,
                ws_bins=[5, 7, 9, 11, 13],
                plot_pars=dict(
                    ax=axes[1],
                    title="Turbine 11: efficiency by wind bin",
                    cmap="magma",
                    vmin=0.4,
                    vmax=1.0,
                ),
            ),
        ],
        farm_results=farm_results,
        ret_dict=True,
    )

fig.axes[-1].set_title("EFF [1]")
fig.tight_layout()
plt.show()
plt.close(fig)

## 3. Horizontal flow field visualizations

Only the wind direction changes between these 10 m/s cases. At 270 degrees, turbines sit directly behind one another; at 240 degrees, wakes pass across the rows. Black segments show rotor positions and orientation. The identical 3-10 m/s colour scales make the two cases comparable.

`FlowPlots2D` requests a 240-by-160 horizontal point grid at the 90 m hub height. Use `calc_outputs` for these flow outputs so the engine can chunk points as well as states; its `states_sel` argument restricts the actual calculation to the requested input labels. With `reduction="states"`, each output returns a list of figures, even for one selected state.

In [ ]:
xy_grid = dict(
    n_img_points=(240, 160),
    xmin=-900,
    xmax=5000,
    ymin=-2200,
    ymax=2200,
    z=90,
)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
with engine:
    for direction, ax in zip([240, 270], axes):
        case = wind_data.index[
            (wind_data[FV.WD] == direction) & (wind_data[FV.WS] == 10)
        ].item()
        flow_figures = algo.calc_outputs(
            outputs=fop.FlowPlots2D(
                var=FV.WS,
                plane="xy",
                reduction="states",
                grid_pars=xy_grid,
                vmin={FV.WS: 3},
                vmax={FV.WS: 10},
                plot_pars=dict(
                    fig=fig,
                    ax=ax,
                    title=f"Wind from {direction} deg | hub height 90 m",
                    cmap="viridis",
                    rotor_color="black",
                    quiver_n=20,
                    quiver_pars=dict(alpha=0.4),
                ),
            ),
            farm_results=farm_results,
            states_sel=[case],
        )

fig.tight_layout()
plt.show()
plt.close(fig)

## 4. Vertical flow field visualizations

Cut through the middle row at `y=0`, with wind from 270 degrees. 

These XZ views use the global eastward x direction (`x_direction=270`). Only rotors intersecting this row are drawn. **The vertical scale is exaggerated** for readability; both axes retain metre labels. The uniform ambient wind has no height shear in this example.

In [ ]:
west_case = wind_data.index[(wind_data[FV.WD] == 270) & (wind_data[FV.WS] == 10)].item()
xz_grid = dict(
    n_img_points=(240, 100),
    xmin=-500,
    xmax=5000,
    zmin=10,
    zmax=400,
    y=0,
    x_direction=270,
)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
with engine:
    for variable, cmap, limits, ax in zip(
        [FV.WS, FV.TI], ["viridis", "inferno"], [(3, 10), (0.06, 0.25)], axes
    ):
        vertical_figures = algo.calc_outputs(
            outputs=fop.FlowPlots2D(
                var=variable,
                plane="xz",
                reduction="states",
                grid_pars=xz_grid,
                vmin={variable: limits[0]},
                vmax={variable: limits[1]},
                plot_pars=dict(
                    fig=fig,
                    ax=ax,
                    title="Wind speed [m/s]"
                    if variable == FV.WS
                    else "Turbulence intensity [1]",
                    cmap=cmap,
                    rotor_color="black",
                ),
            ),
            farm_results=farm_results,
            states_sel=[west_case],
        )
        ax.set_aspect("auto")

fig.tight_layout()
plt.show()
plt.close(fig)

## 5. Visualizing turbine type data and rotors

The NREL 5 MW power/thrust curves explain why a velocity deficit translates into a power loss. A separate **no-wake** wind-speed scan supplies those curves; it must not reuse the waked climate results. `RotorPointPlot` shows the selected rotor points defined when constructing the algorithm that was used in the farm calculation, with coordinates normalized by rotor radius and colours showing integration weights.

In [ ]:
curve_states = foxes.input.states.StatesTable(
    pd.DataFrame({FV.WS: np.linspace(0, 25, 101)}),
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    fixed_vars={FV.WD: 270.0, FV.TI: 0.06, FV.RHO: 1.225},
)
curve_farm = foxes.WindFarm()
curve_farm.add_turbine(
    foxes.Turbine(xy=[0.0, 0.0], turbine_models=["NREL5MW"], name="NREL 5 MW"),
    verbosity=0,
)
curve_algo = foxes.algorithms.Downwind(
    curve_farm, curve_states, wake_models=[], verbosity=0
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
with engine:
    curves = curve_algo.calc_farm(
        outputs=fop.TurbineTypeCurves(
            plot_pars=dict(
                axs=axes[:2],
                titles={FV.P: "Power curve", FV.CT: "Thrust coefficient"},
                linewidth=2,
            )
        ),
        ambient=True,
    )
    rotor_points = algo.calc_outputs(
        outputs=fop.RotorPointPlot(
            plot_pars=dict(ax=axes[2], title=f"Rotor: {algo.rotor_model.name}", s=90)
        ),
        farm_results=farm_results,
    )

fig.tight_layout()
plt.show()
plt.close(fig)

## 6. Flow animations

Now sweep the wind from 210 to 325 degrees at a fixed 10 m/s: **24 frames at 6 fps**. The first movie synchronizes the hub-height velocity field with power traces from turbines 4, 5 and 7 (the middle row). Its lower-panel `P` values are in **kW**, the packaged turbine curve's power unit. The second movie reveals the changing turbulence field.

**These are independent steady solutions, not a simulation of wake travel in time.** The trace panel's `State` axis counts the selected wind cases. For evolving wake histories, see the [dynamic wakes](dyn_wakes.ipynb) and [timelines](timelines.ipynb) notebooks.

Both outputs request the same XY grid, so they can be composed in one point calculation. Each movie has a fixed colour scale. Animations are writer-only outputs: the result mapping is empty, while files are named by appending the variable to `base_name`, here `turning_wind_WS.gif` and `turning_turbulence_TI.gif`.

In [ ]:
movie_cases = wind_data.index[
    (wind_data[FV.WS] == 10) & wind_data[FV.WD].between(210, 325)
].tolist()
with TemporaryDirectory(prefix="foxes-output-gallery-") as movie_dir:
    with engine:
        movie_results = algo.calc_outputs(
            outputs=[
                fop.FlowTurbineAnimation(
                    variables=[FV.WS],
                    turbines=[4, 5, 7],
                    turbine_variable=FV.P,
                    base_name="turning_wind.gif",
                    out_dir=movie_dir,
                    fps=6,
                    figsize=(9, 8),
                    grid_pars=xy_grid,
                    layout_pars=dict(annotate=1, alpha=0, title=""),
                    plot_pars=dict(
                        cmap="viridis",
                        vmin=3,
                        vmax=10,
                        rotor_color="black",
                        title=lambda frame, state: (
                            f"Wind from {wind_data.loc[state, FV.WD]:.0f} deg | 10 m/s"
                        ),
                    ),
                ),
                fop.FlowAnimation(
                    variables=[FV.TI],
                    base_name="turning_turbulence.gif",
                    out_dir=movie_dir,
                    fps=6,
                    figsize=(9, 6),
                    grid_pars=xy_grid,
                    plot_pars=dict(
                        cmap="inferno",
                        vmin=0.06,
                        vmax=0.25,
                        rotor_color="white",
                        title=lambda frame, state: (
                            f"Turbulence | wind from {wind_data.loc[state, FV.WD]:.0f} deg"
                        ),
                    ),
                ),
            ],
            farm_results=farm_results,
            states_sel=movie_cases,
            ret_dict=True,
        )

    for filename in ["turning_wind_WS.gif", "turning_turbulence_TI.gif"]:
        print(filename)
        display(Image(data=(Path(movie_dir) / filename).read_bytes(), format="gif"))

## Make it yours

- Change the turbine spacing or ambient turbulence and rerun: do the alignment losses and wake recovery change as expected?
- Change the movie's direction selection or trace turbines. Increase the grid or frame count only when you need more detail.
- To keep a movie, replace the temporary-directory context with an explicit directory you own and pass it as `out_dir`. `base_name="turning_wind.mp4"` instead creates `turning_wind_WS.mp4`, requiring an installed ffmpeg encoder.
- Need numerical slices, point samples, grouped results or NetCDF writers instead of images? Continue with the [Outputs reference](../outputs.rst). Use separate output calls when requesting different point grids, as the horizontal and vertical examples do here.

The figures and embedded GIFs are the products of this run; all temporary animation files have already been removed. The source notebook is intentionally saved without execution outputs.

## Notes

- This tour is not complete; check the  [Outputs reference](../outputs.rst) and the API. In particular writing results to file without ever constructing the full set of states might be of interest for large cases.
- Here `farm_results` were once fully constructed and then re-used in `calc_outputs`. The full construction of the potentially large `farm_results` Datset object can be avoided by passing outputs directly to `calc_farm(..., outputs=...)`. This would then make use of state chunking wherever possible. However, `calc_farm` interprets point chunking as a loop and is not parallelized in that dimension, in contrast to `calc_outputs(farm_results=..., outputs=...)`.
- Also `calc_points(farm_results=..., points=..., outputs=...)` accepts outputs. However, all outputs that require points (like slice data or flow visualizations) will use the points provided by the `calc_points` parameter and not those generated by the output. Usually this is not intended, so `calc_outputs` or `calc_farm` are to be preferred for outputs with point support.
- Refrain from using multiple outputs with individual (and not coinciding) point support in one `calc_outputs` call, run those seperately instead.